# Scam Call Shield — voice detector training on AMD GPU (ROCm)

Trains BOTH of our HUMAN-vs-AI voice detectors on an **AMD Instinct GPU** via ROCm PyTorch:

1. **CNN baseline** (~300k params, trains in seconds)
2. **Fine-tuned `facebook/wav2vec2-base`** (the production detector — pretrained
   speech representations + our phone-call-augmented data)

Both use a **voice-disjoint split**: 5 TTS voices and 15% of human speakers are
held out entirely, so validation accuracy measures generalization to *unseen* voices.

ROCm PyTorch exposes the GPU through the standard `torch.cuda` API, so this is
the exact same code that runs on NVIDIA — no changes needed.

**Run order:** 1) setup → 2) clone repo → 3) prepare data → 4) train both → 5) verify logs in `voice_trends/`.

In [ ]:
# 1. Environment check — should print an AMD GPU name (e.g. 'AMD Instinct MI300X')
import torch
print('torch      :', torch.__version__)
print('gpu ok     :', torch.cuda.is_available())
print('device     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('is ROCm    :', torch.version.hip is not None, '| hip:', torch.version.hip)

In [ ]:
# If torch is missing or CPU-only on this AMD machine, install ROCm wheels:
# %pip install torch torchaudio --index-url https://download.pytorch.org/whl/rocm6.2
%pip install -q soundfile librosa edge-tts tqdm scikit-learn

In [ ]:
# 2. Get the code (skip if you uploaded the repo manually)
# !git clone https://github.com/Vishalkagade/amd-hackthaon.git
# %cd amd-hackthaon
import os
print('working dir:', os.getcwd())
assert os.path.exists('voice_classifier/train.py'), 'run this notebook from the repo root'

In [ ]:
# 3. Build the dataset: LibriSpeech (human) + edge-tts neural voices (AI)
#    ~15-30 min depending on network. Idempotent — safe to re-run.
!python -m voice_classifier.prepare_data --max-human 1200 --max-ai 1200

In [ ]:
# 4a. Train the CNN baseline on the AMD GPU (seconds). Voice-disjoint split.
!python -m voice_classifier.train --epochs 12 --batch-size 64 --out voice_trends

In [ ]:
# 5. Verify: both logs record the GPU they trained on — AMD-usage evidence.
import json
for name, p in [("CNN baseline", "voice_trends/training_log.json"),
                ("wav2vec2 fine-tune", "voice_trends/wav2vec2_finetuned/finetune_log.json")]:
    log = json.load(open(p))
    print(f"--- {name}")
    print("trained on :", log["gpu"], "| rocm:", log.get("rocm"))
    print("torch      :", log["torch"])
    print("best val acc (voice-disjoint):", log["best_val_acc"])

In [ ]:
# 6. Quick sanity inference — loader prefers the fine-tuned wav2vec2 model
from voice_classifier.infer import load_best_detector
from pathlib import Path
det, name = load_best_detector()
print("using:", name)
h = next(Path('data/human').glob('*.wav')); a = next(Path('data/ai').glob('*.wav'))
print('human clip -> ai_prob =', round(det.score_file(str(h))[0], 3))
print('ai clip    -> ai_prob =', round(det.score_file(str(a))[0], 3))

In [ ]:
# 6. Quick sanity inference on one human + one AI sample
from voice_classifier.infer import VoiceAuthenticity
from pathlib import Path
va = VoiceAuthenticity()
h = next(Path('data/human').glob('*.wav')); a = next(Path('data/ai').glob('*.wav'))
print('human clip -> ai_prob =', round(va.score_file(str(h))[0], 3))
print('ai clip    -> ai_prob =', round(va.score_file(str(a))[0], 3))